In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

current_year = 2026
starting_year = 2012  # Sempre pega os últimos 10 anos

data = []  # Lista para armazenar todas as informações

# Loop para pegar os últimos 10 anos até os próximos 21 anos
for year in range(starting_year, current_year+1):
    for half in ["", "-mid"]:  # Primeiro semestre ("") e segundo semestre ("-mid")
        formatted_year = f"{year}{half}"  # Define o formato correto do ano no URL

        if formatted_year == (str(current_year) + 'mid'):
            continue
            
        url = f"https://www.numbeo.com/quality-of-life/rankings_by_country.jsp?title={formatted_year}&region=150"
        
        response = requests.get(url)

        response.raise_for_status()
        time.sleep(1)

        soup = BeautifulSoup(response.text, 'html.parser')
        
        table = soup.find('tbody')
        if not table:  # Caso a página não tenha dados, pula para o próximo ano
            continue
        
        headers = ["Rank", "Country", "Quality of Life Index", "Purchasing Power Index", 
                   "Safety Index", "Health Care Index", "Cost of Living Index", 
                   "Property Price to Income Ratio", "Traffic Commute Time Index", 
                   "Pollution Index", "Climate Index", "Year"]

        rank = 1  # Inicializa o ranking para cada ano
        
        for row in table.find_all('tr'):
            columns = row.find_all('td')
        
            # Se houver pelo menos uma coluna e a primeira estiver vazia, usá-la como Rank
            if columns and columns[0].text.strip() == "":
                rank_value = rank  # Define o Rank manualmente
                columns = columns[1:]  # Remove a primeira coluna vazia
            else:
                rank_value = rank  # Continua o rank normal
            
            # Ajustar número de colunas para evitar erro de dimensão
            if len(columns) == len(headers) - 2:  # -2 porque adicionamos Rank e Year manualmente
                row_data = [rank_value] + [col.text.strip() for col in columns]  # Adiciona Rank
                
                # Ajusta o ano para exibição correta
                if half == "-mid":
                    row_data.append(f"{year}/2")  # Converte "2015-mid" para "2015/2"
                else:
                    row_data.append(str(year))  # Mantém o ano normal
                
                data.append(row_data)
                rank += 1  # Incrementa o ranking

df = pd.DataFrame(data, columns=headers)

df.to_csv("data/quality_of_life_indices_by_country.csv", index=False)

